# Pilot experiment — augmented copies, max_len, epochs
Same model (`cardiffnlp/twitter-roberta-base-2022-154m`), same GroupKFold folds and seed as the main notebook.
Accuracy is measured on the **original tweets** of folds 0 and 1 (≈4.6k tweets, never used for training).

| config | training data | epochs | max_len |
|---|---|---|---|
| A (current) | originals only | 3 | 96 |
| B | original + 1 augmented copy per ID | 4 | 128 |
| C | original + 2 augmented copies per ID | 3 | 128 |

Colab: Runtime → T4 GPU → Run all (≈25 min). Upload `CAIRLAB.csv` and `test.csv` when asked.

In [ ]:
import os, sys, time, random, subprocess, warnings
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
import numpy as np, pandas as pd, torch, transformers
from sklearn.model_selection import GroupKFold
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
transformers.logging.set_verbosity_error()
SEED = 42
def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); os.environ["PYTHONHASHSEED"] = str(s)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.use_deterministic_algorithms(True, warn_only=DEVICE.type == "cuda")
torch.backends.cudnn.benchmark = False
DATA_DIR = "/content" if IN_COLAB else "."
if IN_COLAB and not os.path.exists(f"{DATA_DIR}/CAIRLAB.csv"):
    from google.colab import files
    print("Please upload CAIRLAB.csv and test.csv"); files.upload()
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

train = pd.read_csv(f"{DATA_DIR}/CAIRLAB.csv", header=None, names=["ID", "Entity", "Sentiment", "Text"])
train["Text"] = train["Text"].fillna("").astype(str)

In [ ]:
CLASSES = np.array(sorted(train.Sentiment.unique()))
train["y"] = train.Sentiment.map({c: i for i, c in enumerate(CLASSES)}).values
train["pos"] = train.groupby("ID").cumcount()
folds = list(GroupKFold(n_splits=5).split(train, train.y, train.ID))
MODEL = "cardiffnlp/twitter-roberta-base-2022-154m"
EPOCH_SCALE = 1       # smoke test only: fewer epochs

def prep(t):
    return " ".join("@user" if w.startswith("@") and len(w) > 1 else "http" if w.startswith("http") else w for w in t.split(" "))

def run(fold, n_aug, epochs, max_len, bs=32, lr=3e-5, freeze=4, smoothing=0.1):
    epochs = max(1, round(epochs * EPOCH_SCALE))
    seed_everything()
    tr_rows, va_rows = folds[fold]
    tr_df = train.iloc[tr_rows]; tr_df = tr_df[tr_df.pos <= n_aug]
    va_df = train.iloc[va_rows]; va_df = va_df[va_df.pos == 0]
    tok = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(CLASSES)).to(DEVICE)
    for p in model.roberta.embeddings.parameters(): p.requires_grad = False
    for layer in model.roberta.encoder.layer[:freeze]:
        for p in layer.parameters(): p.requires_grad = False
    enc = lambda df: tok(df.Entity.tolist(), [prep(t) for t in df.Text], truncation=True, max_length=max_len)["input_ids"]
    def batches(ids, order, size):
        for i in range(0, len(order), size):
            idx = order[i:i + size]; L = max(len(ids[j]) for j in idx)
            x = torch.full((len(idx), L), tok.pad_token_id, dtype=torch.long)
            for r, j in enumerate(idx): x[r, :len(ids[j])] = torch.tensor(ids[j])
            yield idx, x.to(DEVICE), (x != tok.pad_token_id).long().to(DEVICE)
    def bucket_order(ids, rng, size, chunk=50):
        perm = rng.permutation(len(ids)); out = []
        for c in range(0, len(perm), size * chunk):
            part = sorted(perm[c:c + size * chunk], key=lambda j: len(ids[j]))
            out += [part[i:i + size] for i in range(0, len(part), size)]
        rng.shuffle(out); return [j for b in out for j in b]
    use_amp = DEVICE.type == "cuda"
    @torch.inference_mode()
    def acc(ids, y):
        model.eval(); pred = np.zeros(len(ids), dtype=int)
        for idx, x, m in batches(ids, np.argsort([len(v) for v in ids], kind="stable"), 64):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                pred[idx] = model(input_ids=x, attention_mask=m).logits.argmax(-1).cpu().numpy()
        return (pred == y).mean()
    tr_ids, va_ids = enc(tr_df), enc(va_df); ytr = torch.tensor(tr_df.y.values)
    opt = torch.optim.AdamW([q for q in model.parameters() if q.requires_grad], lr=lr, weight_decay=0.01)
    steps = epochs * ((len(tr_ids) + bs - 1) // bs)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=smoothing)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=use_amp)
    rng = np.random.RandomState(SEED); t0 = time.time(); per_epoch = []
    for ep in range(epochs):
        model.train()
        for idx, x, m in batches(tr_ids, bucket_order(tr_ids, rng, bs), bs):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids=x, attention_mask=m).logits
            loss = loss_fn(logits.float(), ytr[idx].to(DEVICE))
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        per_epoch.append(acc(va_ids, va_df.y.values))
    print(f"fold {fold} n_aug={n_aug} epochs={epochs} max_len={max_len} | train rows {len(tr_df)} | "
          f"acc per epoch {np.round(per_epoch, 4).tolist()} | {time.time()-t0:.0f}s", flush=True)
    del model, opt
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return per_epoch[-1], len(va_df)

In [ ]:
CONFIGS = {"A (current)": dict(n_aug=0, epochs=3, max_len=96),
           "B: aug1, 4ep, len128": dict(n_aug=1, epochs=4, max_len=128),
           "C: aug2, 3ep, len128": dict(n_aug=2, epochs=3, max_len=128)}
rows = []
for fold in (0, 1):
    for name, cfg in CONFIGS.items():
        a, n = run(fold, **cfg)
        rows.append(dict(config=name, fold=fold, acc=a, n=n))
res = pd.DataFrame(rows)
summary = res.groupby("config").apply(lambda d: np.average(d.acc, weights=d.n)).rename("acc (folds 0+1)").to_frame()
summary["gain vs A, pt"] = (summary["acc (folds 0+1)"] - summary.loc["A (current)", "acc (folds 0+1)"]) * 100
display(res.pivot(index="config", columns="fold", values="acc").round(4))
display(summary.round(4))
best = summary["gain vs A, pt"].drop("A (current)")
print(f"\nBest variant: {best.idxmax()}  →  {best.max():+.2f} pt")
print("VERDICT:", "worth a full run (gain ≥ 1 pt)" if best.max() >= 1.0 else "NOT worth it (gain < 1 pt) — keep the current submission")